# Bukti GUI Orange 3.40.0: PCA iris dan heart_disease

Bukti dari GUI ada di `evidence/pca/` (setting dan daftar file: `evidence/pca/README.md`).
Aturan uji dan toleransi: `docs/orange-pca-internals.md`. Kode pembanding:
`tests/evidence_io.py` dan `tests/pca_compare.py` (sama dengan yang dipakai pytest).

**Alur bukti**

1. **GUI = skrip.** `.pkl` (skor, Save Data) dan `.npy` (komponen, Python Script) dari
   GUI dibandingkan dengan PCA Orange lewat skrip (projector widget, Normalize ON,
   dataset bawaan yang sama). Aturan uji spec: `np.allclose` default (rtol=1e-5,
   atol=1e-8); komponen seri (heart PC2, PC3) memakai satu tanda s per komponen untuk
   komponen dan skornya; ruang nol heart (PC19-25) hanya skor, rtol=0, atol=1e-8.
   Syaratnya `allclose` dengan toleransi itu. **Observasi** di mesin perekaman
   (Windows, Python Orange yang sama dengan GUI, BLAS sama): selisih maks 0.0,
   bit-identik. Itu fakta pengamatan, bukan syarat: di mesin lain (mis. Colab/Linux,
   BLAS berbeda) bisa muncul selisih ~1e-16 yang tetap lolos toleransi.
2. **CSV hanya pembulatan `%g`.** CSV dari Save Data dibandingkan dengan `.pkl`/`.npy`
   dengan rtol=1e-5, atol=0. `%g` menulis 6 digit signifikan, jadi selisih relatif
   maks harus ≤ 5e-6.
3. **Skrip jadi acuan zestcore.** Karena GUI dan skrip identik dalam toleransi tes, tes
   zestcore cukup membandingkan dengan PCA Orange lewat skrip; GUI tidak perlu dibuka
   lagi.

GUI tidak mengeluarkan `explained_variance_`, hanya rasionya (kolom `variance` di CSV
Components). Aturan uji 1 untuk varians jadi hanya bisa diuji lewat rasio.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import sklearn
import Orange

print("Python      ", sys.version.split()[0])
print("Orange      ", Orange.__version__)
print("NumPy       ", np.__version__)
print("scikit-learn", sklearn.__version__)

# Notebook dijalankan dari notebooks/; semua path relatif ke root repo.
ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "jalankan notebook dari folder notebooks/"
sys.path.insert(0, str(ROOT / "tests"))

# DeprecationWarning dari orangecanvas (paket Orange) saat import OWPCA; bukan dari kode ini.
warnings.filterwarnings("ignore", category=DeprecationWarning, module="orangecanvas")

Python       3.12.12
Orange       3.40.0
NumPy        2.1.3
scikit-learn 1.5.2


**Data dan acuan.** Untuk tiap dataset: PCA Orange lewat skrip (`proj`, skor) dan jumlah
arah varians nol dari rumus aturan uji 2 (sama dengan jumlah terukur, diuji di pytest).

In [2]:
import pandas as pd
from Orange.data import Table

from evidence_io import (read_components_csv, read_components_npy,
                         read_scores_csv, read_scores_pkl)
from pca_compare import (compare, compare_csv, compare_ratio, orange_widget_pca,
                         zero_variance_count)

pd.set_option("display.float_format", "{:.3g}".format)

CASES = [("iris", "iris_pca_norm-on_4pc"),
         ("heart_disease", "heart_pca_norm-on_25pc")]

ACUAN = {}
for dataset, prefix in CASES:
    table = Table(dataset)
    proj, scores = orange_widget_pca(table)
    ACUAN[dataset] = {"prefix": prefix, "proj": proj, "scores": scores,
                      "n_null": zero_variance_count(table)}
    print(f"{dataset}: {len(proj.components_)} komponen, "
          f"{ACUAN[dataset]['n_null']} di ruang nol")


def baris(dataset, perbandingan, jenis, pc, rtol, atol, allclose, maks,
          maks_rel=None, seri="-", catatan=""):
    # Satu baris tabel ringkasan.
    return {"dataset": dataset, "perbandingan": perbandingan, "jenis": jenis,
            "PC": pc, "seri (s)": seri, "rtol": rtol, "atol": atol,
            "allclose": allclose, "selisih maks": maks,
            "selisih rel. maks": maks_rel, "catatan": catatan}

iris: 4 komponen, 0 di ruang nol
heart_disease: 25 komponen, 7 di ruang nol


**1. GUI `.npy`/`.pkl` vs skrip** (komponen dan skor, aturan uji 1-3).

In [3]:
tabel_gui, rincian = [], []
for dataset, a in ACUAN.items():
    rows = compare(a["proj"].components_, a["scores"],
                   read_components_npy(f"{a['prefix']}_components.npy"),
                   read_scores_pkl(f"{a['prefix']}_scores.pkl"), a["n_null"])
    rincian += [{"dataset": dataset, **r} for r in rows]
    k_all = len(a["proj"].components_)
    k = k_all - a["n_null"]
    for nama, jenis, g in [
            ("GUI .npy vs skrip", "komponen", [r for r in rows if r["jenis"] == "komponen"]),
            ("GUI .pkl vs skrip", "skor", [r for r in rows if r["jenis"] == "skor" and r["pc"] <= k]),
            ("GUI .pkl vs skrip", "skor", [r for r in rows if r["jenis"] == "skor" and r["pc"] > k])]:
        if not g:
            continue
        seri = ", ".join(f"PC{r['pc']} ({r['s']:+d})" for r in g if r["seri"]) or "-"
        tabel_gui.append(baris(dataset, nama, jenis, f"PC{g[0]['pc']}-{g[-1]['pc']}",
                               g[0]["rtol"], g[0]["atol"], all(r["allclose"] for r in g),
                               max(r["maks"] for r in g), seri=seri,
                               catatan="ruang nol" if g[0]["pc"] > k else ""))
    if a["n_null"]:
        tabel_gui.append(baris(dataset, "GUI .npy vs skrip", "komponen", f"PC{k + 1}-{k_all}",
                               None, None, None, None,
                               catatan="ruang nol: tidak dibandingkan (aturan uji 2)"))
tabel_gui = pd.DataFrame(tabel_gui)
tabel_gui

,dataset,perbandingan,jenis,PC,seri (s),rtol,atol,allclose,selisih maks,selisih rel. maks,catatan
0,iris,GUI .npy vs skrip,komponen,PC1-4,-,1e-05,1e-08,True,0,None,
1,iris,GUI .pkl vs skrip,skor,PC1-4,-,1e-05,1e-08,True,0,None,
2,heart_disease,GUI .npy vs skrip,komponen,PC1-18,"PC2 (+1), PC3 (+1)",1e-05,1e-08,True,0,None,
3,heart_disease,GUI .pkl vs skrip,skor,PC1-18,"PC2 (+1), PC3 (+1)",1e-05,1e-08,True,0,None,
4,heart_disease,GUI .pkl vs skrip,skor,PC19-25,-,0,1e-08,True,0,None,ruang nol
5,heart_disease,GUI .npy vs skrip,komponen,PC19-25,-,NaN,NaN,None,NaN,None,ruang nol: tidak dibandingkan (aturan uji 2)


**2. Rasio varians: CSV Components (kolom `variance`) vs `explained_variance_ratio_` skrip.**
PC non-nol: pembulatan `%g` (rtol=1e-5, atol=0). Ruang nol: rasio ~1e-16 adalah noise,
aturan yang sama dengan skor ruang nol (rtol=0, atol=1e-8).

In [4]:
tabel_rasio = []
for dataset, a in ACUAN.items():
    _, gui_ratio = read_components_csv(f"{a['prefix']}_components.csv")
    for r in compare_ratio(gui_ratio, a["proj"].explained_variance_ratio_, a["n_null"]):
        tabel_rasio.append(baris(dataset, "CSV rasio vs skrip", "rasio", r["pc"],
                                 r["rtol"], r["atol"], r["allclose"], r["maks"],
                                 r["maks_rel"],
                                 catatan="ruang nol" if r["maks_rel"] is None else ""))
tabel_rasio = pd.DataFrame(tabel_rasio)
tabel_rasio

,dataset,perbandingan,jenis,PC,seri (s),rtol,atol,allclose,selisih maks,selisih rel. maks,catatan
0,iris,CSV rasio vs skrip,rasio,PC1-4,-,1e-05,0,True,4.79e-07,1.01e-06,
1,heart_disease,CSV rasio vs skrip,rasio,PC1-18,-,1e-05,0,True,3.58e-07,3.58e-06,
2,heart_disease,CSV rasio vs skrip,rasio,PC19-25,-,0,1e-08,True,2.9e-22,NaN,ruang nol


**3. CSV skor vs `.pkl`** (file sama dari GUI, beda format; semua elemen termasuk ruang nol).

In [5]:
tabel_csv_skor = []
for dataset, a in ACUAN.items():
    exact = read_scores_pkl(f"{a['prefix']}_scores.pkl")
    r = compare_csv(read_scores_csv(f"{a['prefix']}_scores.csv"), exact)
    tabel_csv_skor.append(baris(dataset, "CSV vs .pkl", "skor", f"PC1-{exact.shape[1]}",
                                r["rtol"], r["atol"], r["allclose"], r["maks"], r["maks_rel"]))
tabel_csv_skor = pd.DataFrame(tabel_csv_skor)
tabel_csv_skor

,dataset,perbandingan,jenis,PC,seri (s),rtol,atol,allclose,selisih maks,selisih rel. maks,catatan
0,iris,CSV vs .pkl,skor,PC1-4,-,1e-05,0,True,4.94e-06,4.82e-06,
1,heart_disease,CSV vs .pkl,skor,PC1-25,-,1e-05,0,True,5e-06,4.95e-06,


**4. CSV komponen vs `.npy`** (semua elemen termasuk ruang nol).

In [6]:
tabel_csv_komp = []
for dataset, a in ACUAN.items():
    exact = read_components_npy(f"{a['prefix']}_components.npy")
    loadings, _ = read_components_csv(f"{a['prefix']}_components.csv")
    r = compare_csv(loadings, exact)
    tabel_csv_komp.append(baris(dataset, "CSV vs .npy", "komponen", f"PC1-{exact.shape[0]}",
                                r["rtol"], r["atol"], r["allclose"], r["maks"], r["maks_rel"]))
tabel_csv_komp = pd.DataFrame(tabel_csv_komp)
tabel_csv_komp

,dataset,perbandingan,jenis,PC,seri (s),rtol,atol,allclose,selisih maks,selisih rel. maks,catatan
0,iris,CSV vs .npy,komponen,PC1-4,-,1e-05,0,True,4.94e-07,1.83e-06,
1,heart_disease,CSV vs .npy,komponen,PC1-25,-,1e-05,0,True,4.99e-07,4.7e-06,


**Tabel perbandingan** (gabungan 1-4).

In [7]:
summary = pd.DataFrame([r for t in (tabel_gui, tabel_rasio, tabel_csv_skor, tabel_csv_komp)
                        for r in t.to_dict("records")])
summary

,dataset,perbandingan,jenis,PC,seri (s),rtol,atol,allclose,selisih maks,selisih rel. maks,catatan
0,iris,GUI .npy vs skrip,komponen,PC1-4,-,1e-05,1e-08,True,0,NaN,
1,iris,GUI .pkl vs skrip,skor,PC1-4,-,1e-05,1e-08,True,0,NaN,
2,heart_disease,GUI .npy vs skrip,komponen,PC1-18,"PC2 (+1), PC3 (+1)",1e-05,1e-08,True,0,NaN,
3,heart_disease,GUI .pkl vs skrip,skor,PC1-18,"PC2 (+1), PC3 (+1)",1e-05,1e-08,True,0,NaN,
4,heart_disease,GUI .pkl vs skrip,skor,PC19-25,-,0,1e-08,True,0,NaN,ruang nol
5,heart_disease,GUI .npy vs skrip,komponen,PC19-25,-,NaN,NaN,None,NaN,NaN,ruang nol: tidak dibandingkan (aturan uji 2)
6,iris,CSV rasio vs skrip,rasio,PC1-4,-,1e-05,0,True,4.79e-07,1.01e-06,
7,heart_disease,CSV rasio vs skrip,rasio,PC1-18,-,1e-05,0,True,3.58e-07,3.58e-06,
8,heart_disease,CSV rasio vs skrip,rasio,PC19-25,-,0,1e-08,True,2.9e-22,NaN,ruang nol
9,iris,CSV vs .pkl,skor,PC1-4,-,1e-05,0,True,4.94e-06,4.82e-06,


**Rincian per PC** (bukti GUI vs skrip). Kolom `s` hanya bisa -1 pada komponen seri.

In [8]:
pd.DataFrame(rincian)

,dataset,jenis,pc,seri,s,rtol,atol,allclose,maks
0,iris,komponen,1,False,1,1e-05,1e-08,True,0
1,iris,skor,1,False,1,1e-05,1e-08,True,0
2,iris,komponen,2,False,1,1e-05,1e-08,True,0
3,iris,skor,2,False,1,1e-05,1e-08,True,0
4,iris,komponen,3,False,1,1e-05,1e-08,True,0
5,iris,skor,3,False,1,1e-05,1e-08,True,0
6,iris,komponen,4,False,1,1e-05,1e-08,True,0
7,iris,skor,4,False,1,1e-05,1e-08,True,0
8,heart_disease,komponen,1,False,1,1e-05,1e-08,True,0
9,heart_disease,skor,1,False,1,1e-05,1e-08,True,0


**Pemeriksaan alur bukti** (sel ini gagal bila salah satu klaim di atas tidak terpenuhi).

In [9]:
diuji = summary.dropna(subset=["allclose"])
assert diuji["allclose"].astype(bool).all()
# Syarat GUI = skrip: allclose. Selisih 0.0 hanya observasi di mesin perekaman.
gui = diuji[diuji["perbandingan"].str.startswith("GUI")]
# Batas %g berlaku untuk setiap baris yang punya selisih relatif (CSV, PC non-nol).
rel = diuji["selisih rel. maks"].dropna()
assert (rel <= 5e-6).all()
print("1. GUI = skrip (allclose): selisih maks teramati", gui["selisih maks"].max())
print("2. CSV vs .pkl/.npy/skrip: selisih rel. maks", f"{rel.max():.3g}", "<= 5e-6")

1. GUI = skrip (allclose): selisih maks teramati 0.0
2. CSV vs .pkl/.npy/skrip: selisih rel. maks 4.95e-06 <= 5e-6
